In [4]:
# ============================================================
# PHASE 1 — DATA IMPORTATION & CLEANING
# Project: Can Technical Indicators Predict the MASI?
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# STEP 1 — LOAD THE RAW DATA
# ============================================================

df = pd.read_csv('Moroccan All Shares Historical Datas.csv')

# Always look at your data first before touching anything
print("=== RAW DATA — First Look ===")
print(df.head(10))
print("\n=== SHAPE ===")
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
print("\n=== COLUMN TYPES ===")
print(df.dtypes)
print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# ============================================================
# PHASE 2 — DATA IMPORTATION & CLEANING
# Project: Can Technical Indicators Predict the MASI?
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# STEP 1 — RELOAD RAW DATA (always start fresh)
# ============================================================
df = pd.read_csv('Moroccan All Shares Historical Datas.csv')

print("=== RAW STATE ===")
print(f"Shape     : {df.shape}")
print(f"Columns   : {df.columns.tolist()}")
print(f"Types     :\n{df.dtypes}")
print(f"\nFirst rows:\n{df.head(3)}")

# ============================================================
# STEP 2 — RENAME COLUMNS (only if raw file has 7 columns)
# ============================================================
if df.shape[1] == 7:
    df.columns = ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Change_pct']
    print("\n✅ Columns renamed")
elif df.shape[1] == 19:
    print("\n⚠️  Looks like you saved the processed file over the raw one.")
    print("    Re-running cleaning on existing processed file — skipping rename.")
else:
    print(f"\n⚠️  Unexpected number of columns: {df.shape[1]}")
    print("    Check your CSV file manually.")

# ============================================================
# STEP 3 — DROP USELESS COLUMNS (if they exist)
# ============================================================
cols_to_drop = [c for c in ['Volume', 'Change_pct'] if c in df.columns]
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"✅ Dropped columns: {cols_to_drop}")

# ============================================================
# STEP 4 — FIX DATE
# ============================================================
# Try multiple formats in case the CSV date format varies
try:
    df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')
except ValueError:
    df['Date'] = pd.to_datetime(df['Date'], infer_datetime_format=True)

print(f"✅ Date parsed — range: {df['Date'].min().date()} → {df['Date'].max().date()}")

# ============================================================
# STEP 5 — FIX NUMERIC COLUMNS (remove commas if present)
# ============================================================
for col in ['Close', 'Open', 'High', 'Low']:
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(',', '', regex=False).astype(float)
        print(f"✅ Fixed commas in: {col}")
    else:
        print(f"   {col} already numeric — skipped")

# ============================================================
# STEP 6 — SORT OLDEST TO NEWEST
# ============================================================
df = df.sort_values('Date').reset_index(drop=True)
print(f"✅ Sorted chronologically")

# ============================================================
# STEP 7 — COMPUTE DAILY RETURN
# ============================================================
df['Return'] = df['Close'].pct_change() * 100

# ============================================================
# STEP 8 — DROP ONLY FIRST ROW (NaN return — no previous day)
# ============================================================
df = df.dropna(subset=['Return']).reset_index(drop=True)

# ============================================================
# STEP 9 — FINAL CHECK
# ============================================================
print("\n=== CLEAN DATA SUMMARY ===")
print(f"Trading days  : {len(df)}")
print(f"Date range    : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Return range  : {df['Return'].min():.2f}% → {df['Return'].max():.2f}%")
print(f"Columns       : {df.columns.tolist()}")
print(f"\n{df.head()}")

# Outlier check — flag suspicious single-day moves
outliers = df[df['Return'].abs() > 5]
if len(outliers) > 0:
    print(f"\n⚠️  {len(outliers)} days with |Return| > 5% (check these):")
    print(outliers[['Date', 'Close', 'Return']].to_string())
else:
    print("\n✅ No extreme outliers found (|Return| > 5%)")

# ============================================================
# STEP 10 — VISUALIZATION
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MASI — Phase 2 Overview', fontsize=14, fontweight='bold')

# Price history
axes[0].plot(df['Date'], df['Close'], color='steelblue', linewidth=1)
axes[0].set_title('MASI Closing Price')
axes[0].set_ylabel('Index Points')
axes[0].set_xlabel('Date')
axes[0].grid(True, alpha=0.3)

# Return distribution
axes[1].hist(df['Return'], bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero')
axes[1].axvline(df['Return'].mean(), color='orange', linestyle='--', linewidth=1.5,
                label=f"Mean: {df['Return'].mean():.3f}%")
axes[1].set_title('Daily Return Distribution')
axes[1].set_xlabel('Return (%)')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('phase2_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Chart saved: phase2_overview.png")

# ============================================================
# STEP 11 — SAVE CLEAN FILE (separate name — don't overwrite raw!)
# ============================================================
df.to_csv('masi_clean.csv', index=False)
print("✅ Saved: masi_clean.csv")
print("\n✅ Phase 2 complete — ready for Phase 3 (Feature Engineering)")

In [ ]:
# ============================================================
# PHASE 2 — FEATURE ENGINEERING (INDICATOR CALCULATION)
# Project: Can Technical Indicators Predict the MASI?
# ============================================================
# ============================================================
# LOAD CLEAN DATA FROM PHASE 1
# ============================================================

df = pd.read_csv('Moroccan All Shares Historical Datas.csv')
df['Date'] = pd.to_datetime(df['Date'])

print("=== DATA LOADED FROM PHASE 1 ===")
print(f"Rows: {df.shape[0]} | Columns: {df.columns.tolist()}")
print(df.head(3))

# ============================================================
# INDICATOR 1 — MOVING AVERAGES (MA7 and MA30)
# MA7  = average of last 7 closing prices  (1 trading week)
# MA30 = average of last 30 closing prices (1 trading month)
# MA_Cross = MA7 - MA30
#   positive → Golden Cross → short term above long term → bullish
#   negative → Death Cross  → short term below long term → bearish
# ============================================================

df['MA7']      = df['Close'].rolling(window=7).mean()
df['MA30']     = df['Close'].rolling(window=30).mean()
df['MA_Cross'] = df['MA7'] - df['MA30']

# ============================================================
# INDICATOR 2 — RSI (14-day window)
# Measures momentum — overbought vs oversold
# Above 70 = overbought (may drop)
# Below 30 = oversold  (may bounce)
# ============================================================

def compute_rsi(series, period=14):
    delta = series.diff()                              # daily price change
    gain  = delta.clip(lower=0).rolling(window=period).mean()   # avg gain
    loss  = -delta.clip(upper=0).rolling(window=period).mean()  # avg loss
    rs    = gain / loss                                # ratio gain/loss
    return 100 - (100 / (1 + rs))                     # scale to 0-100

df['RSI'] = compute_rsi(df['Close'], period=14)

# ============================================================
# INDICATOR 3 — MACD (12, 26, 9)
# Measures momentum direction and strength
# MACD Line   = fast EMA - slow EMA
# Signal Line = smoothed version of MACD Line
# Histogram   = MACD Line - Signal Line (momentum strength)
# ============================================================

ema12             = df['Close'].ewm(span=12, adjust=False).mean()
ema26             = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD']        = ema12 - ema26
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Hist']   = df['MACD'] - df['MACD_Signal']

# ============================================================
# INDICATOR 4 — BOLLINGER BANDS (20-day, 2 std deviations)
# Measures price position relative to its statistical corridor
# BB_Position: 0 = at lower band, 1 = at upper band, 0.5 = middle
# BB_Width:    how wide the corridor is (volatility measure)
# ============================================================

df['BB_Middle']   = df['Close'].rolling(window=20).mean()
df['BB_Std']      = df['Close'].rolling(window=20).std()
df['BB_Upper']    = df['BB_Middle'] + 2 * df['BB_Std']
df['BB_Lower']    = df['BB_Middle'] - 2 * df['BB_Std']
df['BB_Width']    = df['BB_Upper'] - df['BB_Lower']
df['BB_Position'] = (df['Close'] - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower'])

# Drop helper column not needed downstream
df.drop(columns=['BB_Std'], inplace=True)

# ============================================================
# INDICATOR 5 — ATR (14-day Average True Range)
# Measures true daily volatility including overnight gaps
# True Range = max of:
#   High - Low              (day range)
#   |High - Previous Close| (upward gap)
#   |Low  - Previous Close| (downward gap)
# ATR = 14-day average of True Range
# ============================================================

df['H_L']  = df['High'] - df['Low']
df['H_PC'] = abs(df['High'] - df['Close'].shift(1))
df['L_PC'] = abs(df['Low']  - df['Close'].shift(1))
df['TR']   = df[['H_L', 'H_PC', 'L_PC']].max(axis=1)
df['ATR']  = df['TR'].rolling(window=14).mean()

# Drop ATR helper columns
df.drop(columns=['H_L', 'H_PC', 'L_PC', 'TR'], inplace=True)

# ============================================================
# DROP NaN ROWS
# Rolling windows create NaN for the first N rows
# MA30 has the longest window (30) so ~30 rows will be dropped
# This is expected and normal
# ============================================================

rows_before = len(df)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
rows_after = len(df)

print(f"\nRows dropped due to indicator windows: {rows_before - rows_after}")
print(f"Rows remaining: {rows_after}")

# ============================================================
# FINAL VERIFICATION
# ============================================================

print("\n=== PHASE 2 COMPLETE — INDICATOR RANGES ===")
print(f"RSI:         {df['RSI'].min():.2f} → {df['RSI'].max():.2f}   (expected: 0 to 100)")
print(f"MACD:        {df['MACD'].min():.2f} → {df['MACD'].max():.2f}")
print(f"MACD_Hist:   {df['MACD_Hist'].min():.2f} → {df['MACD_Hist'].max():.2f}")
print(f"BB_Position: {df['BB_Position'].min():.2f} → {df['BB_Position'].max():.2f}   (expected: ~0 to 1)")
print(f"BB_Width:    {df['BB_Width'].min():.2f} → {df['BB_Width'].max():.2f}")
print(f"MA_Cross:    {df['MA_Cross'].min():.2f} → {df['MA_Cross'].max():.2f}")
print(f"ATR:         {df['ATR'].min():.2f} → {df['ATR'].max():.2f}")

print("\n=== FINAL COLUMNS ===")
print(df.columns.tolist())

print("\n=== SAMPLE — LAST 5 ROWS ===")
print(df[['Date', 'Close', 'RSI', 'MACD', 'BB_Position', 'MA_Cross', 'ATR']].tail(5))

# ============================================================
# QUICK VISUAL CHECK — All indicators on one figure
# ============================================================

fig, axes = plt.subplots(5, 1, figsize=(14, 18), sharex=True)
fig.suptitle('MASI — Technical Indicators Overview', fontsize=16, fontweight='bold')

# Price + Bollinger Bands + Moving Averages
axes[0].plot(df['Date'], df['Close'],     color='steelblue', linewidth=1,   label='Close')
axes[0].plot(df['Date'], df['MA7'],       color='orange',    linewidth=1,   label='MA7',  linestyle='--')
axes[0].plot(df['Date'], df['MA30'],      color='red',       linewidth=1,   label='MA30', linestyle='--')
axes[0].plot(df['Date'], df['BB_Upper'],  color='gray',      linewidth=0.8, label='BB Upper', linestyle=':')
axes[0].plot(df['Date'], df['BB_Lower'],  color='gray',      linewidth=0.8, label='BB Lower', linestyle=':')
axes[0].fill_between(df['Date'], df['BB_Lower'], df['BB_Upper'], alpha=0.1, color='gray')
axes[0].set_title('Price + Bollinger Bands + Moving Averages')
axes[0].legend(loc='upper left', fontsize=8)
axes[0].grid(True, alpha=0.3)

# RSI
axes[1].plot(df['Date'], df['RSI'], color='purple', linewidth=1)
axes[1].axhline(70, color='red',   linestyle='--', linewidth=0.8, label='Overbought 70')
axes[1].axhline(30, color='green', linestyle='--', linewidth=0.8, label='Oversold 30')
axes[1].axhline(50, color='gray',  linestyle='--', linewidth=0.5)
axes[1].fill_between(df['Date'], 70, df['RSI'], where=(df['RSI'] >= 70), alpha=0.3, color='red')
axes[1].fill_between(df['Date'], 30, df['RSI'], where=(df['RSI'] <= 30), alpha=0.3, color='green')
axes[1].set_title('RSI — Relative Strength Index')
axes[1].set_ylabel('RSI')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# MACD
axes[2].plot(df['Date'], df['MACD'],        color='blue',  linewidth=1, label='MACD Line')
axes[2].plot(df['Date'], df['MACD_Signal'], color='orange',linewidth=1, label='Signal Line')
axes[2].bar(df['Date'],  df['MACD_Hist'],
            color=df['MACD_Hist'].apply(lambda x: 'green' if x >= 0 else 'red'),
            alpha=0.5, label='Histogram')
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_title('MACD — Momentum')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

# BB_Position
axes[3].plot(df['Date'], df['BB_Position'], color='brown', linewidth=1)
axes[3].axhline(1.0, color='red',   linestyle='--', linewidth=0.8, label='Upper Band')
axes[3].axhline(0.0, color='green', linestyle='--', linewidth=0.8, label='Lower Band')
axes[3].axhline(0.5, color='gray',  linestyle='--', linewidth=0.5, label='Middle')
axes[3].fill_between(df['Date'], 0.8, 1.0, alpha=0.1, color='red')
axes[3].fill_between(df['Date'], 0.0, 0.2, alpha=0.1, color='green')
axes[3].set_title('Bollinger Band Position (0=Lower Band, 1=Upper Band)')
axes[3].legend(fontsize=8)
axes[3].grid(True, alpha=0.3)

# ATR
axes[4].plot(df['Date'], df['ATR'], color='darkorange', linewidth=1)
axes[4].set_title('ATR — Average True Range (Volatility)')
axes[4].set_ylabel('Points')
axes[4].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('phase2_indicators.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================
# SAVE
# ============================================================

df.to_csv('Moroccan All Shares Historical Datas.csv', index=False)
print("Saved Moroccan All Shares Historical Data.csv")

In [ ]:
print(f"Total rows: {len(df)}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")

In [ ]:
print(df['Return'].describe())

In [ ]:
# ============================================================
# PHASE 3 — UNIVARIATE ANALYSIS
# ============================================================

from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('Moroccan All Shares Historical Datas.csv')
df['Date'] = pd.to_datetime(df['Date'])

# ============================================================
# STEP 1 — DESCRIPTIVE STATISTICS
# ============================================================

cols = ['Return', 'RSI', 'MACD', 'BB_Width', 'BB_Position', 'MA_Cross', 'ATR']

print("=== DESCRIPTIVE STATISTICS ===")
print(df[cols].describe().round(2))

print("\n=== SKEWNESS & KURTOSIS ===")
for col in cols:
    print(f"{col:15s} | Skewness: {df[col].skew():6.3f} | Kurtosis: {df[col].kurt():6.3f}")

# ============================================================
# STEP 2 — NORMALITY TEST
# ============================================================

jb_stat, jb_pvalue = stats.jarque_bera(df['Return'])
print(f"\n=== JARQUE-BERA TEST ===")
print(f"P-value: {jb_pvalue:.6f} → Returns are NOT normally distributed ✅")

# ============================================================
# STEP 3 — 4 KEY CHARTS ONLY
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('MASI — Univariate Analysis', fontsize=16, fontweight='bold')

# Chart 1 — MASI Price History
axes[0,0].plot(df['Date'], df['Close'], color='steelblue', linewidth=1)
axes[0,0].set_title('MASI Price 2015–2026')
axes[0,0].set_ylabel('Index Points')
axes[0,0].grid(True, alpha=0.3)

# Chart 2 — Return Distribution
axes[0,1].hist(df['Return'], bins=80, color='steelblue', edgecolor='white')
axes[0,1].axvline(0, color='red', linestyle='--', label='Zero')
axes[0,1].axvline(df['Return'].mean(), color='green', linestyle='--', label='Mean')
axes[0,1].set_title('Return Distribution')
axes[0,1].set_xlabel('Return (%)')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Chart 3 — RSI Distribution
axes[1,0].hist(df['RSI'], bins=60, color='purple', edgecolor='white')
axes[1,0].axvline(70, color='red',   linestyle='--', label='Overbought 70')
axes[1,0].axvline(30, color='green', linestyle='--', label='Oversold 30')
axes[1,0].set_title('RSI Distribution')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Chart 4 — ATR over time (shows COVID spike)
axes[1,1].plot(df['Date'], df['ATR'], color='darkorange', linewidth=0.8)
axes[1,1].set_title('ATR — Volatility Over Time (COVID spike visible)')
axes[1,1].set_ylabel('Points')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('phase3_univariate.png', dpi=150)
plt.show()

print("\n✅ Univariate complete — Ready for Bivariate")

In [ ]:
# ============================================================
# PHASE 4 — BIVARIATE ANALYSIS
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('Moroccan All Shares Historical Datas.csv')
df['Date'] = pd.to_datetime(df['Date'])

# ============================================================
# CREATE NEXT DAY RETURN (our target — what we want to predict)
# ============================================================

df['Next_Return'] = df['Return'].shift(-1)
df = df.dropna(subset=['Next_Return'])

# ============================================================
# STEP 1 — CORRELATIONS
# ============================================================

indicators = ['RSI', 'MACD', 'BB_Position', 'MA_Cross', 'ATR']

print("=== CORRELATION WITH NEXT DAY RETURN ===")
for ind in indicators:
    corr = df[ind].corr(df['Next_Return'])
    print(f"{ind:15s} | Correlation: {corr:+.4f}")

# ============================================================
# STEP 2 — RSI ZONE ANALYSIS
# Most important bivariate finding
# ============================================================

df['RSI_Zone'] = pd.cut(df['RSI'],
                         bins=[0, 30, 70, 100],
                         labels=['Oversold', 'Neutral', 'Overbought'])

print("\n=== AVERAGE NEXT DAY RETURN BY RSI ZONE ===")
print(df.groupby('RSI_Zone')['Next_Return'].agg(['mean', 'count']).round(4))

# ============================================================
# STEP 3 — 4 KEY CHARTS
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('MASI — Bivariate Analysis', fontsize=16, fontweight='bold')

# Chart 1 — RSI vs Next Day Return
axes[0,0].scatter(df['RSI'], df['Next_Return'], alpha=0.2, color='purple', s=5)
axes[0,0].axvline(70, color='red',   linestyle='--', linewidth=0.8, label='Overbought')
axes[0,0].axvline(30, color='green', linestyle='--', linewidth=0.8, label='Oversold')
axes[0,0].axhline(0,  color='black', linestyle='--', linewidth=0.5)
corr1 = df['RSI'].corr(df['Next_Return'])
axes[0,0].set_title(f'RSI vs Next Day Return | Corr: {corr1:+.4f}')
axes[0,0].set_xlabel('RSI')
axes[0,0].set_ylabel('Next Day Return (%)')
axes[0,0].legend(fontsize=8)
axes[0,0].grid(True, alpha=0.3)

# Chart 2 — MACD vs Next Day Return
axes[0,1].scatter(df['MACD'], df['Next_Return'], alpha=0.2, color='green', s=5)
axes[0,1].axvline(0, color='red',   linestyle='--', linewidth=0.8, label='Zero line')
axes[0,1].axhline(0, color='black', linestyle='--', linewidth=0.5)
corr2 = df['MACD'].corr(df['Next_Return'])
axes[0,1].set_title(f'MACD vs Next Day Return | Corr: {corr2:+.4f}')
axes[0,1].set_xlabel('MACD')
axes[0,1].legend(fontsize=8)
axes[0,1].grid(True, alpha=0.3)

# Chart 3 — BB_Position vs Next Day Return
axes[1,0].scatter(df['BB_Position'], df['Next_Return'], alpha=0.2, color='brown', s=5)
axes[1,0].axvline(0.8, color='red',   linestyle='--', linewidth=0.8, label='Near Upper')
axes[1,0].axvline(0.2, color='green', linestyle='--', linewidth=0.8, label='Near Lower')
axes[1,0].axhline(0,   color='black', linestyle='--', linewidth=0.5)
corr3 = df['BB_Position'].corr(df['Next_Return'])
axes[1,0].set_title(f'BB Position vs Next Day Return | Corr: {corr3:+.4f}')
axes[1,0].set_xlabel('BB Position (0=Lower, 1=Upper)')
axes[1,0].legend(fontsize=8)
axes[1,0].grid(True, alpha=0.3)

# Chart 4 — ATR vs Absolute Next Return (volatility clustering)
axes[1,1].scatter(df['ATR'], df['Next_Return'].abs(), alpha=0.2, color='darkorange', s=5)
corr4 = df['ATR'].corr(df['Next_Return'].abs())
axes[1,1].set_title(f'ATR vs Next Day Volatility | Corr: {corr4:+.4f}')
axes[1,1].set_xlabel('ATR (today)')
axes[1,1].set_ylabel('Absolute Next Return (%)')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('phase3_bivariate.png', dpi=150)
plt.show()

print("Bivariate complete")

In [ ]:
# ============================================================
# PHASE 3 — MULTIVARIATE ANALYSIS
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

df = pd.read_csv('Moroccan All Shares Historical Datas.csv')

features = ['Return', 'RSI', 'MACD', 'BB_Width', 'BB_Position', 'MA_Cross', 'ATR']

# ============================================================
# STEP 1 — CORRELATION MATRIX
# ============================================================

corr = df[features].corr()

print("=== CORRELATION MATRIX ===")
print(corr.round(2))

# ============================================================
# STEP 2 — PCA
# ============================================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features].dropna())

pca = PCA()
pca.fit(X_scaled)

cumvar = pca.explained_variance_ratio_.cumsum()
n_components_80 = (cumvar >= 0.80).argmax() + 1

print(f"\n=== PCA RESULTS ===")
print(f"Components needed to explain 80% variance: {n_components_80}")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"Component {i+1}: {var*100:.1f}% | Cumulative: {cumvar[i]*100:.1f}%")

# ============================================================
# STEP 3 — 2 CHARTS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('MASI — Multivariate Analysis', fontsize=16, fontweight='bold')

# Chart 1 — Correlation Heatmap
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[0], linewidths=0.5)
axes[0].set_title('Correlation Matrix')

# Chart 2 — PCA Cumulative Variance
axes[1].plot(range(1, len(cumvar)+1), cumvar * 100,
             marker='o', color='steelblue', linewidth=2)
axes[1].axhline(80, color='red', linestyle='--', label='80% threshold')
axes[1].axvline(n_components_80, color='green', linestyle='--',
                label=f'{n_components_80} components')
axes[1].set_title('PCA — Cumulative Explained Variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('phase3_multivariate.png', dpi=150)
plt.show()

print("\n Multivariate complete — Ready for Association Rules")

In [ ]:
import sys
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'mlxtend', '-q'], check=True)

import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

df = pd.read_csv('Moroccan All Shares Historical Datas.csv')
df['Date'] = pd.to_datetime(df['Date'])

# STEP 1 — TARGET VARIABLE
df['Next_Return'] = df['Return'].shift(-1)
df['Target']      = df['Next_Return'].apply(lambda x: 'UP' if x > 0 else 'DOWN')
df = df.dropna(subset=['Next_Return']).reset_index(drop=True)

print(f"UP days   : {(df['Target']=='UP').sum()}")
print(f"DOWN days : {(df['Target']=='DOWN').sum()}")

# STEP 2 — DISCRETIZE
df['RSI_cat']  = pd.cut(df['RSI'],
                         bins=[0, 30, 70, 100],
                         labels=['RSI_Oversold', 'RSI_Neutral', 'RSI_Overbought'])

df['MACD_cat'] = df['MACD'].apply(
    lambda x: 'MACD_Bullish' if x > 0 else 'MACD_Bearish')

df['BB_cat']   = pd.cut(df['BB_Position'],
                         bins=[-999, 0.2, 0.8, 999],
                         labels=['BB_Near_Lower', 'BB_Middle', 'BB_Near_Upper'])

df['ATR_cat']  = pd.qcut(df['ATR'], q=3,
                          labels=['ATR_Low', 'ATR_Med', 'ATR_High'])

df['MA_cat']   = df['MA_Cross'].apply(
    lambda x: 'MA_Golden' if x > 0 else 'MA_Death')

print("\n=== CATEGORY DISTRIBUTION ===")
for col in ['RSI_cat', 'MACD_cat', 'BB_cat', 'ATR_cat', 'MA_cat', 'Target']:
    print(f"\n{col}:")
    print(df[col].value_counts())

# STEP 3 — BUILD TRANSACTIONS
df_rules     = df[['RSI_cat', 'MACD_cat', 'BB_cat',
                    'ATR_cat', 'MA_cat', 'Target']].dropna().astype(str)
transactions = df_rules.values.tolist()

print(f"\nTotal transactions: {len(transactions)}")
print(f"Example: {transactions[0]}")

# STEP 4 — ENCODE
te         = TransactionEncoder()
te_array   = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

print(f"\nEncoded shape: {df_encoded.shape}")

# STEP 5 — RUN APRIORI
# 0.02 instead of 0.05 — finds more valid rules
frequent_itemsets = apriori(df_encoded, min_support=0.02, use_colnames=True)
print(f"\nFrequent itemsets found: {len(frequent_itemsets)}")

# STEP 6 — EXTRACT RULES
rules = association_rules(frequent_itemsets,
                          metric='lift',
                          min_threshold=1.1)

# Keep only rules predicting UP or DOWN
rules = rules[rules['consequents'].astype(str).str.contains('UP|DOWN')]

# Keep only confidence above random
rules = rules[rules['confidence'] > 0.52]

# Sort by confidence
rules = rules.sort_values('confidence', ascending=False).reset_index(drop=True)

print(f"\nTotal prediction rules: {len(rules)}")
print(f"UP rules  : {rules['consequents'].astype(str).str.contains('UP').sum()}")
print(f"DOWN rules: {rules['consequents'].astype(str).str.contains('DOWN').sum()}")

print("\n=== TOP 15 RULES ===")
print(rules[['antecedents', 'consequents', 'support',
             'confidence', 'lift']].head(15).to_string())

# STEP 7 — VISUALIZE
import matplotlib.pyplot as plt

top10 = rules.head(10).copy()
top10['label'] = top10.apply(
    lambda r: str(r['antecedents']).replace('frozenset({','').replace('})','').replace("'",'').strip()
    + ' → ' +
    str(r['consequents']).replace('frozenset({','').replace('})','').replace("'",'').strip(),
    axis=1)

fig, ax = plt.subplots(figsize=(13, 7))
colors = ['green' if 'UP' in r else 'red' for r in top10['label']]
ax.barh(range(len(top10)), top10['confidence'] * 100, color=colors, alpha=0.7)
ax.axvline(50, color='black', linestyle='--', linewidth=1, label='Random baseline 50%')
ax.set_yticks(range(len(top10)))
ax.set_yticklabels(top10['label'], fontsize=8)
ax.set_xlabel('Confidence (%)')
ax.set_title('Top 10 Association Rules — MASI Direction Prediction')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('phase4_association_rules.png', dpi=150)
plt.show()

# SAVE
df.to_csv('masi_with_labels_hibatu.csv', index=False)
print("\n✅ Phase 4 complete")

In [ ]:
# Keep only clean UP or DOWN predictions
pred_rules = rules[
    rules['consequents'].apply(lambda x: x == frozenset({'UP'}) or x == frozenset({'DOWN'}))
].copy()

pred_rules = pred_rules.sort_values('confidence', ascending=False).reset_index(drop=True)

print(f"Clean rules: {len(pred_rules)}")
print(pred_rules[['antecedents','consequents','support','confidence','lift']].head(15).to_string())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv('masi_with_labels_hibatu.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print(f"Shape: {df.shape}")

# ============================================================
# STEP 1 — SELECT FEATURES FOR CLUSTERING
# ============================================================
cluster_features = ['Return', 'RSI', 'MACD', 'BB_Width', 'BB_Position', 'ATR','MA_Cross']
X = df[cluster_features].dropna()

# ============================================================
# STEP 2 — SCALE DATA
# ============================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ============================================================
# STEP 3 — ELBOW METHOD
# ============================================================
inertias = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertias, marker='o', color='steelblue', linewidth=2)
plt.title('Elbow Method — Optimal Number of Clusters', fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('elbow.png', dpi=150)
plt.show()

# ============================================================
# STEP 4 — APPLY K-MEANS K=3
# ============================================================
km_final = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = km_final.fit_predict(X_scaled)

# ============================================================
# STEP 5 — AUTO LABEL CLUSTERS
# ============================================================
# Bull = highest return / Crisis = highest ATR / Bear = rest
cluster_means = df.groupby('Cluster')[['Return', 'ATR', 'RSI']].mean()
print("\nCluster means:")
print(cluster_means.round(3))

bull   = cluster_means['Return'].idxmax()
crisis = cluster_means['ATR'].idxmax()
bear   = [x for x in [0,1,2] if x != bull and x != crisis][0]

cluster_names = {bull: 'Bull Market', crisis: 'Crisis', bear: 'Bear Market'}
df['Market_Regime'] = df['Cluster'].map(cluster_names)

print(f"\nBull   = Cluster {bull}")
print(f"Crisis = Cluster {crisis}")
print(f"Bear   = Cluster {bear}")
print("\nRegime distribution:")
print(df['Market_Regime'].value_counts())

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# ============================================================
# STEP 1 — SELECT FEATURES
# Using exact column names from your notebook
# ============================================================
cluster_features = ['Return', 'ATR', 'RSI', 'BB_Width']

df_cluster = df[cluster_features + ['Date', 'Close']].dropna().copy()
print(f"Days for clustering: {len(df_cluster)}")

# ============================================================
# STEP 2 — SCALE
# ============================================================
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster[cluster_features])

# ============================================================
# STEP 3 — ELBOW METHOD
# ============================================================
inertias = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(range(2, 9), inertias, marker='o', color='steelblue', linewidth=2)
plt.axvline(3, color='red', linestyle='--', label='Chosen k=3')
plt.title('Elbow Method — Optimal Number of Clusters', fontweight='bold')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('phase5_elbow.png', dpi=150)
plt.show()

# ============================================================
# STEP 4 — APPLY K-MEANS K=3
# ============================================================
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_cluster['Cluster'] = kmeans.fit_predict(X_scaled)

print("\n=== CLUSTER SIZES ===")
print(df_cluster['Cluster'].value_counts().sort_index())

# ============================================================
# STEP 5 — AUTO LABEL
# Highest Return = Bull / Highest ATR = Crisis / Rest = Bear
# ============================================================
cluster_means = df_cluster.groupby('Cluster')[cluster_features].mean()
print("\n=== CLUSTER MEANS ===")
print(cluster_means.round(3))

bull   = cluster_means['Return'].idxmax()
crisis = cluster_means['ATR'].idxmax()
bear   = [x for x in [0, 1, 2] if x != bull and x != crisis][0]

names = {bull: 'Bull', crisis: 'Crisis', bear: 'Bear'}
df_cluster['Regime'] = df_cluster['Cluster'].map(names)

print(f"\nBull   = Cluster {bull}")
print(f"Crisis = Cluster {crisis}")
print(f"Bear   = Cluster {bear}")

print("\n=== REGIME DISTRIBUTION ===")
print(df_cluster['Regime'].value_counts())

# ============================================================
# STEP 6 — PLAIN LANGUAGE SUMMARY
# ============================================================
print("\n=== REGIME CHARACTERISTICS ===")
for regime in ['Bull', 'Bear', 'Crisis']:
    s = df_cluster[df_cluster['Regime'] == regime]
    print(f"\n{regime} ({len(s)} days):")
    print(f"  Avg Return  : {s['Return'].mean():+.4f}%")
    print(f"  Avg ATR     : {s['ATR'].mean():.1f}")
    print(f"  Avg RSI     : {s['RSI'].mean():.1f}")
    print(f"  Avg BB_Width: {s['BB_Width'].mean():.1f}")

# ============================================================
# STEP 7 — PRICE CHART COLORED BY REGIME
# ============================================================
color_map = {'Bull': 'green', 'Bear': 'red', 'Crisis': 'orange'}

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('MASI Market Regimes 2015-2026 (K-Means, k=3)',
             fontsize=14, fontweight='bold')

for regime, color in color_map.items():
    mask = df_cluster['Regime'] == regime
    axes[0].scatter(
        df_cluster.loc[mask, 'Date'],
        df_cluster.loc[mask, 'Close'],
        c=color, s=4, label=regime, alpha=0.7
    )
axes[0].set_title('MASI Price — Colored by Market Regime')
axes[0].set_ylabel('Index Points')
axes[0].legend(markerscale=3, fontsize=10)
axes[0].grid(True, alpha=0.3)

regime_numeric = df_cluster['Regime'].map({'Bull': 1, 'Bear': 0, 'Crisis': -1})
for regime, color, val in [('Bull','green',1), ('Bear','red',0), ('Crisis','orange',-1)]:
    mask = df_cluster['Regime'] == regime
    axes[1].fill_between(df_cluster['Date'], regime_numeric,
                         where=mask, color=color, alpha=0.4, label=regime)
axes[1].set_title('Regime Timeline')
axes[1].set_ylabel('Regime')
axes[1].set_yticks([-1, 0, 1])
axes[1].set_yticklabels(['Crisis', 'Bear', 'Bull'])
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('phase5_regimes.png', dpi=150)
plt.show()

# ============================================================
# STEP 8 — PIE CHART
# ============================================================
regime_counts = df_cluster['Regime'].value_counts()
colors_pie    = [color_map[r] for r in regime_counts.index]

plt.figure(figsize=(7, 7))
plt.pie(regime_counts, labels=regime_counts.index,
        colors=colors_pie, autopct='%1.1f%%',
        startangle=90, textprops={'fontsize': 12})
plt.title('MASI — Days per Market Regime 2015-2026', fontsize=13)
plt.tight_layout()
plt.savefig('phase5_pie.png', dpi=150)
plt.show()

# ============================================================
# STEP 9 — MERGE REGIME BACK TO MAIN df
# Required for Step 6 prediction model
# ============================================================
df = df.merge(df_cluster[['Date', 'Regime']], on='Date', how='left')

print("\n=== REGIME ADDED TO df ===")
print(df['Regime'].value_counts())
print(f"\n✅ Clustering complete — Ready for Prediction Model")

In [ ]:
# ============================================================
# PHASE 6 — SUPERVISED MODEL
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# LOAD
df = pd.read_csv('masi_with_labels_hibatu.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ============================================================
# STEP 1 — TARGET VARIABLE
# ============================================================

df['Target'] = (df['Return'].shift(-1) > 0).astype(int)
df = df.dropna(subset=['Target']).reset_index(drop=True)

baseline = df['Target'].mean()
print(f"UP days  : {df['Target'].sum()} ({baseline*100:.1f}%)")
print(f"DOWN days: {(df['Target']==0).sum()} ({(1-baseline)*100:.1f}%)")
print(f"Baseline : {max(baseline, 1-baseline)*100:.1f}%")

# ============================================================
# STEP 2 — FEATURES AND TIME SPLIT
# ============================================================

features = ['RSI', 'MACD', 'MACD_Hist', 'BB_Width',
            'BB_Position', 'MA_Cross', 'ATR', 'Return']

X = df[features]
y = df['Target']

split    = int(len(df) * 0.8)
X_train  = X.iloc[:split]
X_test   = X.iloc[split:]
y_train  = y.iloc[:split]
y_test   = y.iloc[split:]

print(f"\nTrain: {df['Date'].iloc[0].date()} → {df['Date'].iloc[split].date()} ({len(X_train)} days)")
print(f"Test : {df['Date'].iloc[split].date()} → {df['Date'].iloc[-1].date()} ({len(X_test)} days)")

# Scale — fit on train only
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ============================================================
# STEP 3 — LOGISTIC REGRESSION
# ============================================================

lr      = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)
lr_acc  = accuracy_score(y_test, lr_pred)

print("\n=== LOGISTIC REGRESSION ===")
print(f"Accuracy : {lr_acc*100:.2f}%")
print(classification_report(y_test, lr_pred, target_names=['DOWN','UP']))

# ============================================================
# STEP 4 — RANDOM FOREST
# ============================================================

rf      = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)
rf_acc  = accuracy_score(y_test, rf_pred)

print("\n=== RANDOM FOREST ===")
print(f"Accuracy : {rf_acc*100:.2f}%")
print(classification_report(y_test, rf_pred, target_names=['DOWN','UP']))

# ============================================================
# STEP 5 — FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame({
    'Feature'   : features,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n=== FEATURE IMPORTANCE ===")
print(importance.to_string(index=False))

# ============================================================
# STEP 6 — VISUALIZE
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('MASI — Supervised Model Results', fontsize=14, fontweight='bold')

# Chart 1 — Model Comparison
models     = ['Baseline', 'Logistic Reg', 'Random Forest']
accuracies = [max(baseline, 1-baseline)*100, lr_acc*100, rf_acc*100]
colors     = ['gray', 'steelblue', 'green']
axes[0].bar(models, accuracies, color=colors, alpha=0.8)
axes[0].axhline(max(baseline,1-baseline)*100, color='gray', linestyle='--')
axes[0].set_ylim(45, 70)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Model Comparison')
for i, (acc, color) in enumerate(zip(accuracies, colors)):
    axes[0].text(i, acc + 0.3, f'{acc:.1f}%', ha='center', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Chart 2 — Confusion Matrix Random Forest
cm = confusion_matrix(y_test, rf_pred)
axes[1].imshow(cm, cmap='Greens')
axes[1].set_xticks([0,1]); axes[1].set_yticks([0,1])
axes[1].set_xticklabels(['DOWN','UP'])
axes[1].set_yticklabels(['DOWN','UP'])
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
axes[1].set_title('Random Forest — Confusion Matrix')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, cm[i,j], ha='center', va='center',
                     fontsize=14, fontweight='bold',
                     color='white' if cm[i,j] > cm.max()/2 else 'black')

# Chart 3 — Feature Importance
axes[2].barh(importance['Feature'], importance['Importance'],
             color='steelblue', alpha=0.8)
axes[2].set_xlabel('Importance')
axes[2].set_title('Feature Importance')
axes[2].invert_yaxis()
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('phase6_results.png', dpi=150)
plt.show()

print("\n✅ Phase 6 complete — Ready for Walk-Forward Validation")

In [ ]:
# ============================================================
# PHASE 7 — WALK-FORWARD VALIDATION
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# LOAD
df = pd.read_csv('masi_with_labels_hibatu.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

df['Target'] = (df['Return'].shift(-1) > 0).astype(int)
df = df.dropna(subset=['Target']).reset_index(drop=True)

features = ['RSI', 'MACD', 'MACD_Hist', 'BB_Width',
            'BB_Position', 'MA_Cross', 'ATR', 'Return']

# ============================================================
# WALK-FORWARD LOOP
# Train on everything before test year
# Test on that specific year only
# ============================================================

test_years = [2022, 2023, 2024, 2025]
results    = []

print("=== WALK-FORWARD VALIDATION ===\n")

for year in test_years:

    # Split
    train = df[df['Date'].dt.year < year]
    test  = df[df['Date'].dt.year == year]

    if len(test) == 0:
        continue

    X_train = train[features]
    y_train = train['Target']
    X_test  = test[features]
    y_test  = test['Target']

    # Scale
    scaler         = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # Train Random Forest
    rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
    rf.fit(X_train_scaled, y_train)
    pred = rf.predict(X_test_scaled)

    acc      = accuracy_score(y_test, pred)
    baseline = y_test.mean()
    baseline = max(baseline, 1 - baseline)

    results.append({
        'Year'    : year,
        'Accuracy': acc,
        'Baseline': baseline,
        'Days'    : len(test)
    })

    print(f"{year} | Accuracy: {acc*100:.2f}% | Baseline: {baseline*100:.2f}% | Days: {len(test)}")

# ============================================================
# VISUALIZE
# ============================================================

results_df = pd.DataFrame(results)

x     = range(len(results_df))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar([i - width/2 for i in x], results_df['Accuracy'] * 100,
       width, label='Random Forest', color='steelblue', alpha=0.8)
ax.bar([i + width/2 for i in x], results_df['Baseline'] * 100,
       width, label='Baseline',      color='gray',      alpha=0.6)

ax.set_xticks(list(x))
ax.set_xticklabels(results_df['Year'])
ax.set_ylabel('Accuracy (%)')
ax.set_title('Walk-Forward Validation — Accuracy per Year vs Baseline')
ax.legend()
ax.set_ylim(45, 70)
ax.grid(True, alpha=0.3, axis='y')

for i, row in results_df.iterrows():
    ax.text(i - width/2, row['Accuracy'] * 100 + 0.3,
            f"{row['Accuracy']*100:.1f}%", ha='center', fontsize=9, fontweight='bold')
    ax.text(i + width/2, row['Baseline'] * 100 + 0.3,
            f"{row['Baseline']*100:.1f}%", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('phase7_walkforward.png', dpi=150)
plt.show()

print("\n✅ Walk-Forward Validation complete")

In [ ]:
# ============================================================
# PHASE 7 — WALK-FORWARD VALIDATION
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ============================================================
# STEP 1 — PREPARE
# Same features and target as Phase 6
# ============================================================
features = ['RSI', 'MACD', 'MACD_Hist', 'BB_Width',
            'BB_Position', 'MA_Cross', 'ATR', 'Return']

df['Target'] = (df['Return'].shift(-1) > 0).astype(int)
df_wf = df[features + ['Target', 'Date']].dropna().copy()
df_wf = df_wf.sort_values('Date').reset_index(drop=True)

print(f"Total days : {len(df_wf)}")
print(f"Range      : {df_wf['Date'].min().date()} to {df_wf['Date'].max().date()}")

# ============================================================
# STEP 2 — WALK-FORWARD LOOP
# Train on everything BEFORE year → Test on that year only
# ============================================================
test_years = [2022,2023, 2024, 2025]
results    = []

print("\n=== WALK-FORWARD VALIDATION ===")
print(f"{'Year':<8} {'Train Days':>12} {'Test Days':>11} {'Accuracy':>10} {'Baseline':>10} {'Edge':>8}")
print("-" * 62)

for year in test_years:
    train = df_wf[df_wf['Date'].dt.year <  year]
    test  = df_wf[df_wf['Date'].dt.year == year]

    if len(test) < 10:
        print(f"  {year} — not enough data")
        continue

    X_train = train[features]
    y_train = train['Target']
    X_test  = test[features]
    y_test  = test['Target']

    # Scale — fit on train only, apply to test
    scaler         = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # Train fresh model for this year
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        random_state=42
    )
    model.fit(X_train_scaled, y_train)
    preds    = model.predict(X_test_scaled)

    acc      = accuracy_score(y_test, preds)
    baseline = y_test.mean()
    edge     = acc - baseline

    results.append({
        'Year'      : year,
        'Train_Days': len(train),
        'Test_Days' : len(test),
        'Accuracy'  : acc,
        'Baseline'  : baseline,
        'Edge'      : edge,
        'y_test'    : y_test,
        'preds'     : preds
    })

    print(f"  {year:<6} {len(train):>12} {len(test):>11} {acc:>9.1%} {baseline:>9.1%} {edge:>+7.1%}")

# ============================================================
# STEP 3 — DETAILED REPORT PER YEAR
# ============================================================
print("\n=== CLASSIFICATION REPORT PER YEAR ===")
for r in results:
    print(f"\n--- {r['Year']} (trained on 2015 to {r['Year']-1}) ---")
    print(f"  Accuracy : {r['Accuracy']:.1%}")
    print(f"  Baseline : {r['Baseline']:.1%}")
    print(f"  Edge     : {r['Edge']:+.1%}")
    print(classification_report(r['y_test'], r['preds'],
                                target_names=['DOWN', 'UP'],
                                zero_division=0))

# ============================================================
# STEP 4 — SUMMARY TABLE
# ============================================================
df_results = pd.DataFrame([{k: v for k, v in r.items()
                             if k not in ['y_test','preds']}
                            for r in results])

print("\n=== SUMMARY ===")
print(f"  Average Accuracy : {df_results['Accuracy'].mean():.1%}")
print(f"  Average Baseline : {df_results['Baseline'].mean():.1%}")
print(f"  Average Edge     : {df_results['Edge'].mean():+.1%}")
print(f"  Positive years   : {(df_results['Edge'] > 0).sum()} / {len(df_results)}")

# ============================================================
# STEP 5 — VISUALIZATION
# ============================================================
years     = df_results['Year'].astype(int).tolist()
accs      = (df_results['Accuracy'] * 100).tolist()
baselines = (df_results['Baseline'] * 100).tolist()
edges     = (df_results['Edge'] * 100).tolist()
sizes     = df_results['Train_Days'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('MASI — Walk-Forward Validation (2022 | 2023 | 2024 | 2025)',
             fontsize=13, fontweight='bold')

# Chart 1 — Accuracy vs Baseline per year
width = 0.35
x     = range(len(years))
bars1 = axes[0].bar([i - width/2 for i in x], accs,
                     width, label='Model', color='steelblue', alpha=0.8)
axes[0].bar([i + width/2 for i in x], baselines,
             width, label='Baseline', color='gray', alpha=0.6)
axes[0].axhline(50, color='black', linestyle='--', linewidth=1, label='Random 50%')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(years)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Accuracy vs Baseline per Year')
axes[0].set_ylim(40, 70)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3, axis='y')
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3,
                 f'{bar.get_height():.1f}%',
                 ha='center', fontsize=9, fontweight='bold')

# Chart 2 — Edge per year
colors_edge = ['green' if e > 0 else 'red' for e in edges]
axes[1].bar(range(len(years)), edges, color=colors_edge, alpha=0.8, edgecolor='white')
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_xticks(range(len(years)))
axes[1].set_xticklabels(years)
axes[1].set_ylabel('Edge (%)')
axes[1].set_title('Edge Over Baseline\n(green = beats baseline)')
axes[1].grid(True, alpha=0.3, axis='y')
for i, edge in enumerate(edges):
    axes[1].text(i, edge + 0.2 if edge >= 0 else edge - 0.5,
                 f'{edge:+.1f}%', ha='center',
                 fontsize=10, fontweight='bold')

# Chart 3 — Training size growth
axes[2].bar(range(len(years)), sizes,
            color='steelblue', alpha=0.7, edgecolor='white')
axes[2].set_xticks(range(len(years)))
axes[2].set_xticklabels(years)
axes[2].set_ylabel('Training Days')
axes[2].set_title('Training Set Size per Year\n(grows each round)')
axes[2].grid(True, alpha=0.3, axis='y')
for i, size in enumerate(sizes):
    axes[2].text(i, size + 10, str(size),
                 ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('phase7_walkforward.png', dpi=150)
plt.show()

print("\n✅ Walk-Forward complete — Project finished")

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('masi_with_labels_hibatu.csv')
df['Date'] = pd.to_datetime(df['Date'])

# Recreate regimes
cluster_features = ['Return', 'RSI', 'MACD', 'BB_Width', 'BB_Position', 'ATR', 'MA_Cross']
X = df[cluster_features].dropna()

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans        = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

cluster_means = df.groupby('Cluster')[['Return', 'ATR']].mean()
bull          = cluster_means['Return'].idxmax()
crisis        = cluster_means['ATR'].idxmax()
bear          = [x for x in [0,1,2] if x != bull and x != crisis][0]

df['Regime'] = df['Cluster'].map({bull: 'Bull', crisis: 'Crisis', bear: 'Bear'})

print(df['Regime'].value_counts())

df.to_csv('masi_final_hibatu.csv', index=False)
print("✅ Saved to masi_final.csv")

In [ ]:
import pandas as pd

# STEP 1 — Load and inspect
df = pd.read_csv('masi_final_hibatu.csv')  # her filename

print("=== SHAPE ===")
print(df.shape)

print("\n=== COLUMNS ===")
print(df.columns.tolist())

print("\n=== FIRST ROW ===")
print(df.iloc[0].to_dict())

print("\n=== DATA TYPES ===")
print(df.dtypes)

print("\n=== NULL VALUES ===")
print(df.isnull().sum())

In [ ]:
# ============================================================
# CONVERT CSV TO JSON FOR MASI DASHBOARD
# ============================================================
import pandas as pd
import numpy as np

# STEP 1 — LOAD
df = pd.read_csv('masi_final_hibatu.csv')
df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')
df = df.sort_values('Date').reset_index(drop=True)

# STEP 2 — FIX TARGET: "UP"/"DOWN" → 1/0
df['Target'] = df['Target'].map({'UP': 1, 'DOWN': 0})

print("Target distribution:")
print(df['Target'].value_counts())

print("\nRegime distribution:")
print(df['Regime'].value_counts())

# STEP 3 — ROUND FLOATS
float_cols = df.select_dtypes(include=[np.float64]).columns
df[float_cols] = df[float_cols].round(4)

# STEP 4 — EXPORT
df.to_json('masi_data_hibatu.json', orient='records')
print(f"\n✅ Done! {len(df)} rows exported to masi_data.json")

In [ ]:
import json
import numpy as np

# ==============================
# BASELINE
# ==============================
baseline = max(
    (df['Target'] == 'UP').mean(),
    (df['Target'] == 'DOWN').mean()
) * 100


# ==============================
# REGIME STATISTICS
# ==============================
regime_stats = {}

for regime in df['Regime'].unique():
    subset = df[df['Regime'] == regime]

    regime_stats[str(regime)] = {
        "avg_return": f"{subset['Return'].mean():+.2f}%",
        "avg_atr":    f"{subset['ATR'].mean():.2f}",
        "avg_rsi":    f"{subset['RSI'].mean():.1f}",
    }


# ==============================
# FEATURE IMPORTANCE (RandomForest)
# ==============================
features = ['RSI', 'MACD', 'MACD_Hist', 'BB_Width',
            'BB_Position', 'MA_Cross', 'ATR', 'Return']

feat_imp = [
    {"feature": f, "importance": round(imp * 100, 1)}
    for f, imp in sorted(
        zip(features, rf.feature_importances_),
        key=lambda x: x[1],
        reverse=True
    )
]


# ==============================
# TOP 5 CLEAN RULES
# ==============================
top_rules = []

for _, row in pred_rules.head(5).iterrows():
    antecedents = ' + '.join(list(row['antecedents']))
    consequents = ' + '.join(list(row['consequents']))

    top_rules.append({
        "rule": f"{antecedents} → {consequents}",
        "confidence": round(float(row['confidence']) * 100, 1),
        "lift": round(float(row['lift']), 2),
        "support": round(float(row['support']) * 100, 2),
    })


# ==============================
# RULE COUNTS
# ==============================
rules_total = len(rules)
rules_up    = pred_rules['consequents'].astype(str).str.contains('UP').sum()
rules_down  = pred_rules['consequents'].astype(str).str.contains('DOWN').sum()


# ==============================
# WALK-FORWARD RESULTS
# ==============================
wf_data = []

for r in results:
    wf_data.append({
        "year":     str(r['Year']),
        "accuracy": round(r['Accuracy'] * 100, 1),
        "baseline": round(r['Baseline'] * 100, 1),
    })

best_acc = max(wf_data, key=lambda x: x['accuracy'])['accuracy']


# ==============================
# FINAL JSON STRUCTURE
# ==============================
model_results = {
    "baseline_accuracy": round(baseline, 1),
    "logistic_regression_accuracy": round(lr_acc * 100, 1),
    "random_forest_accuracy": round(rf_acc * 100, 1),
    "best_walkforward_accuracy": best_acc,

    "association_rules": {
        "total_rules_found": rules_total,
        "clean_prediction_rules": len(pred_rules),
        "rules_predicting_up": int(rules_up),
        "rules_predicting_down": int(rules_down),
        "top_5_rules": top_rules
    },

    "feature_importance_random_forest": feat_imp,
    "walk_forward_validation": wf_data,
    "market_regime_statistics": regime_stats
}


# ==============================
# SAVE JSON FILE
# ==============================
with open('model_results.json', 'w') as f:
    json.dump(model_results, f, indent=2)

print("✅ model_results.json saved successfully!")
print(json.dumps(model_results, indent=2))